In [ ]:
print("HELLO WORLD")

HELLO WORLD


In [9]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import jax
import jax.numpy as jnp

import wavefunctions, trajectory, folx_hamiltonian
from complex_vmc import c_wavefunctions, c_trajectory, c_folx_hamiltonian, c_optimization

import importlib

In [3]:
jax.config.update("jax_enable_x64", True)

In [69]:
N = 14
r_ws = 10.0
walkers = 16
dim = 3
numKpoints = N

spins = (N//2,N-N//2)
lattice = wavefunctions.computeLattice(N, r_ws, dim)
kpoints = wavefunctions.genKpoints(numKpoints, lattice, dim)[:spins[0]]

rng = jax.random.PRNGKey(558)
rng, rs_rng, init_rng = jax.random.split(rng, 3)

centers = trajectory.generateBCC(spins, lattice, dim)
(upCenters,downCenters) = centers[:spins[0]], centers[spins[0]:]
rs = centers[None,:,:] + (r_ws / 10) * jax.random.normal(rs_rng, shape=(walkers,N,dim))

wavefunction = wavefunctions.LogSimpleSlaters(spins, dim, kpoints)
wavefunction = wavefunctions.LogSlaterCYJastrow(spins, dim, lattice, kpoints)
mala = trajectory.MALAUpdater(wavefunction, r_ws)
kinetic_energy = folx_hamiltonian.LocalKineticEnergy(wavefunction, 6)

parameters = wavefunction.initBatch(init_rng, rs)

logmags = wavefunction.applyBatch(parameters, rs)
newRs = mala.updateBatch(parameters, rs, rng, 1e-1)
kes = kinetic_energy.batch(parameters, rs)

ref_ke = jnp.sum(jnp.square(kpoints))

print(logmags.shape)
print(newRs.shape)
print(kes.shape)
print()

print(ref_ke)
print(kes)

(16,)
(16, 14, 3)
(16,)

0.15692780148560848
[1.45828114 1.73804371 2.18244267 1.94173111 1.93791003 1.75552926
 2.22695446 1.95698646 1.74126278 2.02101617 3.99961197 3.22517035
 2.0999522  1.23008572 2.08478441 2.72808987]


In [17]:
importlib.reload(c_wavefunctions)
importlib.reload(c_trajectory)
importlib.reload(c_folx_hamiltonian)
importlib.reload(c_optimization)

N = 14
r_ws = 10.0
walkers = 16
dim = 3
numKpoints = N

spins = (N//2,N-N//2)
lattice = c_wavefunctions.computeLattice(N, r_ws, dim)
kpoints = c_wavefunctions.genKpoints(numKpoints, lattice, dim)[:spins[0]]

rng = jax.random.PRNGKey(558)
rng, rs_rng, init_rng = jax.random.split(rng, 3)

centers = c_trajectory.generateBCC(spins, lattice, dim)
(upCenters,downCenters) = centers[:spins[0]], centers[spins[0]:]
rs = centers[None,:,:] + (r_ws / 10) * jax.random.normal(rs_rng, shape=(walkers,N,dim))

#wavefunction = c_wavefunctions.LogSimpleSlaters(spins, dim, kpoints)
wavefunction = c_wavefunctions.LogSlaterCYJastrow(spins, dim, lattice, kpoints)
mala = c_trajectory.MALAUpdater(wavefunction, r_ws)
energy = c_folx_hamiltonian.LocalEnergyUEG3D(wavefunction, lattice, 6)
optimizer = c_optimization.StochasticReconfigurationMomentum(wavefunction, energy)

parameters = wavefunction.initBatch(init_rng, rs)

phases, logmags = wavefunction.applyBatch(parameters, rs)
newRs = mala.updateBatch(parameters, rs, rng, 1e-1)
energies = energy.batch(parameters, rs)

ref_ke = jnp.sum(jnp.square(kpoints))

print(phases.shape, logmags.shape)
print(phases.dtype, logmags.dtype)
print(newRs.shape)
print(energies.shape)
print()

print(parameters)

optimizer(parameters, rs, 1e-2, 1e-3, 0.0, 0.0)

(16,) (16,)
float64 float64
(16, 14, 3)
(16,)

{'params': {'CYJastrow': {'As_same_diff': Array([18.25741858, 18.25741858], dtype=float64)}}}


float64
[-0.00105116 -0.00720698]


Exception: Debugging...